# Pipeline Orquestador — Arquitectura Medallion StackOverflow

**Proyecto 3 — Arquitectura Lakehouse en Azure Databricks**

Este notebook ejecuta el pipeline completo **Bronze → Silver → Gold** en secuencia.

## Uso
- **Ejecución manual:** correr todas las celdas en orden
- **Databricks Job:** este notebook es la tarea principal del Job con 3 tasks encadenadas

## Notebooks ejecutados
| Orden | Notebook | Capa | Descripción |
|---|---|---|---|
| 1 | `bronze_ingest` | Bronze | Ingesta desde S3 → Parquet override en ADLS |
| 2 | `silver_transform` | Silver | Transformación → Delta con MERGE |
| 3 | `gold_agg` | Gold | Agregaciones y KPIs → Delta con MERGE |

In [0]:
# ============================================================
# CELDA 1 — Configuración de rutas
# Ajustar NOTEBOOK_BASE a la carpeta donde están tus notebooks
# ============================================================
NOTEBOOK_BASE = "/Workspace/Proyecto3"

BRONZE_NB = f"{NOTEBOOK_BASE}/bronze_ingest"
SILVER_NB = f"{NOTEBOOK_BASE}/silver_transform (1)"
GOLD_NB   = f"{NOTEBOOK_BASE}/gold_agg (1)"

TIMEOUT = 7200  # 2 horas por notebook

print(f"[CONFIG] Bronze:  {BRONZE_NB}")
print(f"[CONFIG] Silver:  {SILVER_NB}")
print(f"[CONFIG] Gold:    {GOLD_NB}")
print(f"[CONFIG] Timeout: {TIMEOUT}s por etapa")


[CONFIG] Bronze:  /Workspace/Proyecto3/bronze_ingest
[CONFIG] Silver:  /Workspace/Proyecto3/silver_transform (1)
[CONFIG] Gold:    /Workspace/Proyecto3/gold_agg (1)
[CONFIG] Timeout: 7200s por etapa


In [0]:
# ============================================================
# CELDA 2 — Ejecución del pipeline completo
# ============================================================
import time

pipeline_start = time.time()
results = {}

print("=" * 60)
print("PIPELINE MEDALLION — Bronze → Silver → Gold")
print("=" * 60)

# ── ETAPA 1: Bronze ──────────────────────────────────────────
print("\n[1/3] Iniciando Bronze...")
t0 = time.time()
try:
    result_bronze = dbutils.notebook.run(BRONZE_NB, timeout_seconds=TIMEOUT)
    elapsed = time.time() - t0
    results["bronze"] = "OK"
    print(f"[OK] Bronze completado en {elapsed:.0f}s")
except Exception as e:
    results["bronze"] = f"ERROR: {e}"
    print(f"[ERROR] Bronze falló: {e}")
    raise  # detener el pipeline si Bronze falla

# ── ETAPA 2: Silver ──────────────────────────────────────────
print("\n[2/3] Iniciando Silver...")
t0 = time.time()
try:
    result_silver = dbutils.notebook.run(SILVER_NB, timeout_seconds=TIMEOUT)
    elapsed = time.time() - t0
    results["silver"] = "OK"
    print(f"[OK] Silver completado en {elapsed:.0f}s")
except Exception as e:
    results["silver"] = f"ERROR: {e}"
    print(f"[ERROR] Silver falló: {e}")
    raise  # detener el pipeline si Silver falla

# ── ETAPA 3: Gold ────────────────────────────────────────────
print("\n[3/3] Iniciando Gold...")
t0 = time.time()
try:
    result_gold = dbutils.notebook.run(GOLD_NB, timeout_seconds=TIMEOUT)
    elapsed = time.time() - t0
    results["gold"] = "OK"
    print(f"[OK] Gold completado en {elapsed:.0f}s")
except Exception as e:
    results["gold"] = f"ERROR: {e}"
    print(f"[ERROR] Gold falló: {e}")
    raise

# ── Reporte final ─────────────────────────────────────────────
total = time.time() - pipeline_start
print("\n" + "=" * 60)
print("RESUMEN DEL PIPELINE")
print("=" * 60)
for etapa, status in results.items():
    icon = "✓" if status == "OK" else "✗"
    print(f"  {icon} {etapa.upper():<10} {status}")
print(f"\n[DONE] Pipeline completado en {total:.0f}s ({total/60:.1f} min)")
print("Catálogo: lacm_uao_prod_central_us")
print("Capas:    bronze | silver | gold")


PIPELINE MEDALLION — Bronze → Silver → Gold

[1/3] Iniciando Bronze...
[OK] Bronze completado en 404s

[2/3] Iniciando Silver...
[OK] Silver completado en 273s

[3/3] Iniciando Gold...
[OK] Gold completado en 222s

RESUMEN DEL PIPELINE
  ✓ BRONZE     OK
  ✓ SILVER     OK
  ✓ GOLD       OK

[DONE] Pipeline completado en 899s (15.0 min)
Catálogo: lacm_uao_prod_central_us
Capas:    bronze | silver | gold
